# Semana 3 · Datos reales: pandas, gráficos y estadística descriptiva

**Nivel Intermedio** · Semana 3 · Laboratorio [B] · 2 horas · CPU (no hace falta GPU)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/melvinpqbsc/Alto_Rendimiento_IA/blob/main/Nivel_Intermedio/clases/sem03_pandas.ipynb)

**Objetivos.** Al terminar el laboratorio podrás:

1. Cargar un CSV con pandas, leer sus tipos de columnas y encontrar y tratar los datos faltantes.
2. Resumir una variable con media, mediana y desviación estándar, y comparar grupos con `groupby`.
3. Hacer histogramas y diagramas de dispersión que respondan una pregunta, y medir una relación con la correlación.
4. Ver que una imagen y un texto también terminan siendo arreglos de números.

**Prerrequisitos.** Semana 2: arreglos de NumPy, máscaras, `axis` y `argmax`. Lectura previa: *Data Analysis with Pandas* y *Descriptive statistics* de MÓLÓ. El laboratorio no repite la lectura: la pone a trabajar con datos reales.

**Cómo funciona.** Es un laboratorio: casi todo son ejercicios, de menor a mayor dificultad. Cada uno tiene una celda que lo revisa (✓, ✗ o "todavía sin resolver"). Avanza a tu ritmo; el profesor va pasando por los grupos. Los ejercicios 8 y 9 son para quien llegue.

**Agenda** (después de la apertura de 15 minutos).

| Parte | Tema | Minutos |
|---|---|---|
| 1 | Cargar un CSV: filas, columnas y tipos | 10 |
| 2 | Seleccionar y filtrar | 10 |
| 3 | Datos faltantes | 10 |
| 4 | Estadística descriptiva: media, mediana, desviación | 15 |
| 5 | Comparar grupos con `groupby` | 15 |
| 6 | Gráficos: histogramas | 10 |
| 7 | Dispersión y correlación | 15 |
| 8 | De un gráfico a una regla (con línea base) | 10 |
| 9 | Datos estructurados vs no estructurados | 5 |
| 10 | Cierre | 5 |

Ejecuta esta celda primero: carga las librerías, fija la semilla y define `check`, la función que revisa tus ejercicios.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits

SEED = 0
rng = np.random.default_rng(SEED)

URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"


def check(name, test):
    """Ejecuta una prueba y muestra ✓, ✗ o "todavía sin resolver", sin detener el notebook."""
    try:
        test()
        print(f"✓ {name}")
    except NotImplementedError:
        print(f"… {name}: todavía sin resolver")
    except AssertionError as e:
        print(f"✗ {name}: {e}")
    except Exception as e:
        print(f"✗ {name}: {type(e).__name__}: {e}")


print("pandas", pd.__version__)

## 1. Cargar un CSV

Un **CSV** (*comma-separated values*) es una tabla guardada como texto: una fila por línea y las columnas separadas por comas. Es el formato más común de los datos tabulares, y el de muchos problemas de competencia.

Usamos **Palmer Penguins**: 344 pingüinos de tres especies (*Adelie*, *Chinstrap*, *Gentoo*) medidos en tres islas de la Antártida. pandas lo lee en una línea y lo guarda en un ***DataFrame***: una tabla con nombre en cada columna. Cada columna es una ***Series***, un arreglo de NumPy con etiquetas.

En *machine learning*, cada **fila** es un ejemplo (un pingüino) y cada **columna** es una variable (*feature*).

In [ ]:
df = pd.read_csv(URL)
print("forma:", df.shape)
df.head()

Lo primero con una tabla nueva es mirar los **tipos** de sus columnas. `float64` es numérica; `object` casi siempre es texto, es decir, una variable **categórica**: toma unos pocos valores fijos (la especie, la isla, el sexo).

In [ ]:
df.info()

### Ejercicio 1: conocer la tabla

1. `n_rows` y `n_cols`: cuántas filas y columnas tiene `df`. Pista: `df.shape`.
2. `numeric_cols`: la lista de los nombres de las columnas numéricas, en el orden de la tabla. Pista: `df.select_dtypes("number").columns`, convertido con `list(...)`.
3. `categorical_cols`: la lista de las columnas de texto (`"object"`).

In [ ]:
n_rows = None            # TODO
n_cols = None            # TODO
numeric_cols = None      # TODO
categorical_cols = None  # TODO

In [ ]:
def test_table():
    if any(v is None for v in [n_rows, n_cols, numeric_cols, categorical_cols]):
        raise NotImplementedError
    assert (n_rows, n_cols) == (344, 7), f"(n_rows, n_cols) = {(n_rows, n_cols)}"
    assert list(numeric_cols) == ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"], \
        f"numeric_cols = {numeric_cols}"
    assert list(categorical_cols) == ["species", "island", "sex"], f"categorical_cols = {categorical_cols}"


check("Conocer la tabla", test_table)

## 2. Seleccionar y filtrar

Lo mismo que en NumPy, con nombres en vez de posiciones:

| Expresión | Qué devuelve |
|---|---|
| `df["species"]` | una columna (una `Series`) |
| `df[["species", "body_mass_g"]]` | varias columnas (un `DataFrame`) |
| `df[df["body_mass_g"] > 5000]` | las filas que cumplen la condición: una **máscara**, como en la semana 2 |
| `df[(cond1) & (cond2)]` | dos condiciones a la vez. Con `&` (y), `\|` (o) y **paréntesis** en cada una |
| `df.loc[3, "species"]` | una celda: fila con etiqueta 3, columna `"species"` |

Y para las categóricas, `value_counts()` cuenta cuántas veces aparece cada valor.

In [ ]:
print(df["species"].value_counts())
heavy_gentoo = df[(df["species"] == "Gentoo") & (df["body_mass_g"] > 5500)]
print("Gentoo de más de 5,5 kg:", len(heavy_gentoo))
heavy_gentoo[["species", "island", "body_mass_g", "sex"]].head()

### Ejercicio 2: filtrar

1. `n_gentoo`: cuántos pingüinos *Gentoo* hay.
2. `n_heavy`: cuántos pingüinos pesan más de 5000 g (`body_mass_g > 5000`).
3. `biscoe_share`: qué fracción de los pingüinos vive en la isla *Biscoe*, como número entre 0 y 1. Pista: el promedio de una máscara es la fracción de `True`.
4. `longest_flipper_species`: la especie del pingüino con la aleta (*flipper*) más larga. Pista: `idxmax()` devuelve la etiqueta de la fila del máximo; luego `df.loc[...]`.

In [ ]:
n_gentoo = None                 # TODO
n_heavy = None                  # TODO
biscoe_share = None             # TODO
longest_flipper_species = None  # TODO

In [ ]:
def test_filter():
    if any(v is None for v in [n_gentoo, n_heavy, biscoe_share, longest_flipper_species]):
        raise NotImplementedError
    assert n_gentoo == 124, f"n_gentoo = {n_gentoo}"
    assert n_heavy == 61, f"n_heavy = {n_heavy}; ¿usaste > 5000?"
    assert 0 <= biscoe_share <= 1, "biscoe_share debe ser una fracción entre 0 y 1"
    assert np.isclose(biscoe_share, 168 / 344), f"biscoe_share = {biscoe_share:.3f}"
    assert longest_flipper_species == "Gentoo", f"longest_flipper_species = {longest_flipper_species!r}"


check("Filtrar", test_filter)

## 3. Datos faltantes

Los datos reales tienen huecos (*missing values*): un sensor que falló, una medida que nadie anotó. pandas los marca con `NaN` (*not a number*). Antes de calcular nada hay que saber **cuántos hay y dónde**.

In [ ]:
print(df.isna().sum())
df[df.isna().any(axis=1)]

Dos pingüinos (las filas 3 y 339) no tienen ninguna medida, y a otros nueve solo les falta el sexo. Hay tres salidas, y elegir es parte del análisis:

| Opción | Código | Cuándo |
|---|---|---|
| Borrar las filas | `df.dropna(subset=[...])` | quedan pocas incompletas y no tienen nada útil |
| Rellenar con un valor | `df["col"].fillna(valor)` | la mediana para un número, `"desconocido"` para una categoría |
| Dejarlos | — | si la columna no se va a usar |

**Cuidado**: pandas ignora los `NaN` al calcular (`df["body_mass_g"].mean()` funciona), pero NumPy no: un solo `NaN` contamina todo el resultado.

In [ ]:
print("pandas:", df["body_mass_g"].mean())
print("NumPy: ", np.mean(df["body_mass_g"].to_numpy()))

### Ejercicio 3: tratar los faltantes

1. `missing`: una `Series` con la cantidad de faltantes de cada columna.
2. `n_complete`: cuántas filas no tienen **ningún** faltante.
3. `df_clean`: `df` sin las filas a las que les falta alguna medida **numérica** (usa `numeric_cols` del ejercicio 1, o escribe la lista). Las filas a las que solo les falta el sexo se quedan.
4. `sex_filled`: la columna `sex` de `df_clean` con los faltantes reemplazados por `"UNKNOWN"`.

In [ ]:
missing = None     # TODO
n_complete = None  # TODO
df_clean = None    # TODO
sex_filled = None  # TODO

In [ ]:
def test_missing():
    if any(v is None for v in [missing, n_complete, df_clean, sex_filled]):
        raise NotImplementedError
    assert missing["sex"] == 11 and missing["body_mass_g"] == 2, "missing debe contar los NaN de cada columna"
    assert n_complete == 333, f"n_complete = {n_complete}"
    assert df_clean.shape == (342, 7), f"df_clean tiene forma {df_clean.shape}, esperaba (342, 7)"
    assert df_clean["body_mass_g"].isna().sum() == 0, "a df_clean todavía le faltan medidas"
    assert len(sex_filled) == 342 and sex_filled.isna().sum() == 0, "sex_filled no debe tener NaN"
    assert (sex_filled == "UNKNOWN").sum() == 9, f"esperaba 9 'UNKNOWN', hay {(sex_filled == 'UNKNOWN').sum()}"


check("Datos faltantes", test_missing)

Desde aquí trabajamos con la tabla limpia, `penguins`: la misma idea del ejercicio 3, escrita aquí para que el resto del laboratorio funcione aunque no lo hayas terminado.

In [ ]:
NUMERIC = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
penguins = df.dropna(subset=NUMERIC).copy()
penguins["sex"] = penguins["sex"].fillna("UNKNOWN")
print(penguins.shape)

## 4. Estadística descriptiva

Tres números resumen una variable:

- **Media** (*mean*): $\bar{x} = \frac{1}{n}\sum_i x_i$, el punto de equilibrio.
- **Mediana** (*median*): el valor del medio cuando los ordenas. La mitad de los datos queda por debajo.
- **Desviación estándar** (*standard deviation*): cuánto se alejan los datos de la media, en las mismas unidades que los datos.

$$s = \sqrt{\frac{1}{n-1}\sum_i (x_i - \bar{x})^2}$$

¿Por qué $n - 1$ y no $n$? Con una muestra, $n-1$ corrige un pequeño sesgo (se llama corrección de Bessel). pandas divide por $n-1$ y NumPy por $n$, salvo que le pidas otra cosa con `ddof`. Con 342 datos la diferencia es mínima, pero un `assert` que compare los dos fallará.

`describe()` da todo de una vez, incluidos los cuartiles: el 25 % de los datos queda por debajo de `25%`.

In [ ]:
penguins[NUMERIC].describe().round(1)

### Ejercicio 4: estadística a mano

1. Completa `my_mean(x)` y `my_std(x)` **con NumPy**, con las fórmulas de arriba (la desviación divide por $n-1$). Sin `np.mean`, `np.std` ni los métodos de pandas: solo sumas, `len` y `np.sqrt`.
2. Un **valor atípico** (*outlier*). Alguien anotó mal un peso: 380 000 g en vez de 3800 g. Calcula `mean_bad` y `median_bad`, la media y la mediana de `masses_bad`. ¿Cuál se movió más respecto de los datos buenos?

In [ ]:
def my_mean(x):
    x = np.asarray(x, dtype=float)
    # TODO
    raise NotImplementedError


def my_std(x):
    x = np.asarray(x, dtype=float)
    # TODO
    raise NotImplementedError


masses = penguins["body_mass_g"].to_numpy()
masses_bad = masses.copy()
masses_bad[0] = 380_000   # un error de tipeo: dos ceros de más

mean_bad = None    # TODO
median_bad = None  # TODO

In [ ]:
def test_stats():
    x = penguins["flipper_length_mm"]
    assert np.isclose(my_mean(x), x.mean()), "my_mean no coincide con pandas"
    assert np.isclose(my_std(x), x.std()), "my_std no coincide con pandas; ¿dividiste por n - 1?"
    assert np.isclose(my_std([2, 4, 4, 4, 5, 5, 7, 9]), 2.138, atol=1e-3), "my_std([2, 4, 4, 4, 5, 5, 7, 9]) ≈ 2.138"


def test_outlier():
    if mean_bad is None or median_bad is None:
        raise NotImplementedError
    assert np.isclose(mean_bad, masses_bad.mean()), "mean_bad debe ser la media de masses_bad"
    assert np.isclose(median_bad, np.median(masses_bad)), "median_bad debe ser la mediana de masses_bad"


check("Media y desviación", test_stats)
check("Valor atípico", test_outlier)
print(f"datos buenos: media {masses.mean():7.0f} g · mediana {np.median(masses):5.0f} g")
if mean_bad is not None:
    print(f"con el error: media {mean_bad:7.0f} g · mediana {median_bad:5.0f} g")

Un solo dato malo arrastró la media unos 1100 g; la mediana ni se movió. La mediana es **robusta** a los valores atípicos. Por eso, ante una media rara, lo primero es buscar el máximo y el mínimo: `describe()` los muestra.

## 5. Comparar grupos con `groupby`

"¿Los *Gentoo* pesan más que los *Adelie*?" es la pregunta más común sobre una tabla. `groupby` la responde en tres pasos: **separa** las filas por el valor de una columna, **aplica** una función a cada grupo y **junta** los resultados en una tabla.

In [ ]:
print(penguins.groupby("species")["body_mass_g"].mean().round(0))
penguins.groupby("species")[["flipper_length_mm", "body_mass_g"]].agg(["mean", "std"]).round(1)

Con dos columnas en `groupby` salen los grupos cruzados. `unstack()` pasa el segundo nivel a columnas, y la tabla queda más fácil de leer:

In [ ]:
penguins[penguins["sex"] != "UNKNOWN"].groupby(["species", "sex"])["body_mass_g"].mean().unstack().round(0)

### Ejercicio 5: grupos

1. `flipper_by_species`: la longitud media de la aleta de cada especie (una `Series` con las especies como índice).
2. `longest_species`: el nombre de la especie con la aleta media más larga. Pista: `idxmax()`.
3. `counts`: cuántos pingüinos hay de cada especie en cada isla, como `DataFrame` con las islas como filas y las especies como columnas (0 donde no hay ninguno). Pista: `groupby([...]).size().unstack(fill_value=0)`, o `pd.crosstab`.
4. Mira `counts`. `only_adelie_island`: el nombre de la isla donde solo viven *Adelie*.

In [ ]:
flipper_by_species = None  # TODO
longest_species = None     # TODO
counts = None              # TODO
only_adelie_island = None  # TODO

In [ ]:
def test_groups():
    if any(v is None for v in [flipper_by_species, longest_species, counts, only_adelie_island]):
        raise NotImplementedError
    assert np.isclose(flipper_by_species["Adelie"], 189.95, atol=0.01), "flipper_by_species['Adelie'] ≈ 189.95"
    assert longest_species == "Gentoo", f"longest_species = {longest_species!r}"
    assert counts.loc["Biscoe", "Gentoo"] == 123, "counts.loc['Biscoe', 'Gentoo'] debe ser 123"
    assert counts.loc["Dream", "Gentoo"] == 0, "pon 0 donde no hay pingüinos (fill_value=0)"
    assert only_adelie_island == "Torgersen", "mira la fila de cada isla en counts"


check("Grupos", test_groups)
counts if counts is not None else None

## 6. Gráficos: histogramas

Un número resume; un gráfico muestra la **forma** de los datos. Un **histograma** (*histogram*) parte el rango de una variable en intervalos (*bins*) y cuenta cuántos datos caen en cada uno.

Toda figura de este curso lleva **ejes con nombre y unidades** y una **leyenda** si hay más de un grupo. Un gráfico sin unidades no se puede leer.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].hist(penguins["body_mass_g"], bins=25, color="gray")
axes[0].set_xlabel("masa corporal (g)")
axes[0].set_ylabel("cantidad de pingüinos")
axes[0].set_title("Todos juntos")

for species in ["Adelie", "Chinstrap", "Gentoo"]:
    group = penguins[penguins["species"] == species]
    axes[1].hist(group["body_mass_g"], bins=20, alpha=0.6, label=species)
axes[1].set_xlabel("masa corporal (g)")
axes[1].set_title("Separados por especie")
axes[1].legend()
plt.show()

El histograma de la izquierda tiene dos "jorobas". El de la derecha explica por qué: los *Gentoo* son mucho más pesados, y *Adelie* y *Chinstrap* casi se superponen. Una media de todos juntos (4200 g) describe un pingüino que casi no existe.

### Ejercicio 6: tu histograma

Haz un histograma del largo del pico (`bill_length_mm`) con una barra de color por especie, como el de la derecha, en los ejes `ax`. Ponle nombre (con unidades) a los dos ejes, un título y una leyenda.

Después, mira el gráfico y responde en la celda de texto: ¿qué especie se separa de las otras por el pico? ¿Cuál se separaba por el peso?

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
# TODO: un ax.hist(...) por especie, con label=...
# TODO: ax.set_xlabel, ax.set_ylabel, ax.set_title, ax.legend
plt.show()

In [ ]:
def test_histogram():
    if not ax.containers:
        raise NotImplementedError
    assert len(ax.containers) == 3, f"esperaba un histograma por especie (3), hay {len(ax.containers)}"
    assert ax.get_xlabel() and ax.get_ylabel(), "ponle nombre a los dos ejes"
    assert "(" in ax.get_xlabel(), "el eje x necesita unidades, por ejemplo (mm)"
    assert ax.get_title(), "falta el título"
    assert ax.get_legend() is not None, "falta la leyenda"


check("Histograma", test_histogram)

**Tu respuesta:** …

## 7. Dispersión y correlación

Un **diagrama de dispersión** (*scatter plot*) pone un punto por fila, con una variable en cada eje. Muestra si dos variables se mueven juntas.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for species in ["Adelie", "Chinstrap", "Gentoo"]:
    group = penguins[penguins["species"] == species]
    ax.scatter(group["flipper_length_mm"], group["body_mass_g"], s=12, alpha=0.7, label=species)
ax.set_xlabel("largo de la aleta (mm)")
ax.set_ylabel("masa corporal (g)")
ax.legend()
plt.show()

Aleta más larga, pingüino más pesado: los puntos suben hacia la derecha. La **correlación** de Pearson (*correlation*) mide esa tendencia con un número entre −1 y 1:

$$r = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_i (x_i - \bar{x})^2}\,\sqrt{\sum_i (y_i - \bar{y})^2}}$$

- $r$ cerca de 1: suben juntas, casi en línea recta. Cerca de −1: cuando una sube, la otra baja. Cerca de 0: no hay relación **lineal**.
- El numerador suma productos de desvíos: un punto arriba a la derecha de las medias (los dos desvíos positivos) suma, uno arriba a la izquierda resta.
- **Correlación no es causalidad.** Que dos variables se muevan juntas no dice cuál causa a cuál, ni si las dos dependen de una tercera.

`df.corr()` calcula la correlación de cada par de columnas:

In [ ]:
penguins[NUMERIC].corr().round(2)

### Ejercicio 7: correlación a mano y una sorpresa

1. Completa `my_corr(x, y)` con NumPy usando la fórmula de arriba (sin `np.corrcoef` ni `.corr()`). Pista: primero resta las medias.
2. `r_all`: la correlación entre `bill_length_mm` y `bill_depth_mm` en **todos** los pingüinos, con tu `my_corr`.
3. `r_by_species`: un diccionario `{especie: r}` con la misma correlación calculada **dentro** de cada especie.

Compara el signo de `r_all` con los de `r_by_species`, y después mira el gráfico de la celda siguiente.

In [ ]:
def my_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    # TODO
    raise NotImplementedError


r_all = None         # TODO
r_by_species = None  # TODO: {"Adelie": ..., "Chinstrap": ..., "Gentoo": ...}

In [ ]:
def test_my_corr():
    x, y = rng.normal(size=50), rng.normal(size=50)
    assert np.isclose(my_corr(x, y), np.corrcoef(x, y)[0, 1]), "my_corr no coincide con np.corrcoef"
    assert np.isclose(my_corr(x, 3 * x + 1), 1.0), "la correlación de x con 3x + 1 debe ser 1"


def test_simpson():
    if r_all is None or r_by_species is None:
        raise NotImplementedError
    assert np.isclose(r_all, -0.235, atol=1e-3), f"r_all = {r_all:.3f}"
    assert set(r_by_species) == {"Adelie", "Chinstrap", "Gentoo"}, "r_by_species necesita una entrada por especie"
    assert np.isclose(r_by_species["Gentoo"], 0.643, atol=1e-3), f"r_by_species['Gentoo'] = {r_by_species['Gentoo']:.3f}"


check("my_corr", test_my_corr)
check("Correlación por especie", test_simpson)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].scatter(penguins["bill_length_mm"], penguins["bill_depth_mm"], s=12, color="gray")
axes[0].set_title("Todos juntos")
for species, group in penguins.groupby("species"):
    axes[1].scatter(group["bill_length_mm"], group["bill_depth_mm"], s=12, alpha=0.7, label=species)
axes[1].set_title("Por especie")
axes[1].legend()
for a in axes:
    a.set_xlabel("largo del pico (mm)")
axes[0].set_ylabel("alto del pico (mm)")
plt.show()

Juntos, la correlación es negativa: picos más largos parecen más bajos. Dentro de **cada** especie es positiva: un pico más largo también es más alto. El signo se invierte porque los *Gentoo* tienen picos largos y bajos, y mezclarlos con los demás inclina la nube entera. Esto se llama **paradoja de Simpson**, y la lección es útil para todo el año: **antes de confiar en un número de toda la tabla, míralo por grupos.**

## 8. De un gráfico a una regla

Los histogramas sugieren un clasificador sin entrenar nada: la aleta separa a los *Gentoo* y el pico separa a los *Chinstrap* de los *Adelie*. Antes de probarlo, la **línea base**: predecir siempre la especie más común.

In [ ]:
majority = penguins["species"].value_counts().idxmax()
baseline_acc = (penguins["species"] == majority).mean()
print(f"línea base (siempre {majority}): accuracy {baseline_acc:.3f}")

El *accuracy* (exactitud) es la fracción de aciertos. La línea base acierta el 44 %: cualquier regla que valga la pena tiene que superarla con holgura.

### Ejercicio 8: una regla hecha a mano

Completa `predict_rule(data)`: devuelve un arreglo con una especie por fila de `data`, con esta regla de dos preguntas:

1. Si la aleta mide más de `FLIPPER_CUT` mm → `"Gentoo"`.
2. Si no, si el pico mide más de `BILL_CUT` mm → `"Chinstrap"`.
3. Si no → `"Adelie"`.

Elige los dos umbrales mirando los histogramas de las partes 6 y 7. Pista: `np.where(condición, si_verdadero, si_falso)` funciona como un `if` sobre todo el arreglo, y se puede anidar. La meta es superar el 90 % de *accuracy*.

In [ ]:
FLIPPER_CUT = None  # TODO: un número, en mm
BILL_CUT = None     # TODO: un número, en mm


def predict_rule(data):
    # TODO
    raise NotImplementedError

In [ ]:
def test_rule():
    if FLIPPER_CUT is None or BILL_CUT is None:
        raise NotImplementedError
    pred = np.asarray(predict_rule(penguins))
    assert pred.shape == (len(penguins),), f"esperaba una predicción por fila, forma {pred.shape}"
    acc = (pred == penguins["species"].to_numpy()).mean()
    print(f"línea base:      accuracy {baseline_acc:.3f}")
    print(f"regla a mano:    accuracy {acc:.3f}")
    assert acc > 0.9, f"accuracy {acc:.3f}; mueve los umbrales mirando los histogramas"


check("Regla a mano", test_rule)

Dos umbrales y más del 90 % de aciertos. Pero ojo: elegiste los umbrales **mirando los mismos datos** con los que mides el *accuracy*. Con pingüinos nuevos, probablemente acertarías un poco menos. Medir bien, con datos que el modelo no vio, es el tema de la semana 5.

## 9. Datos estructurados vs no estructurados

Los pingüinos son **datos estructurados** (*structured data*): una tabla en la que cada columna significa algo ("largo del pico"). Las imágenes, los textos y el audio son **datos no estructurados**: ningún número suelto significa nada por sí solo. Pero para un modelo, al final, **todo es un arreglo de números**.

**Una imagen** ya la viste en la semana 2: un dígito de 8×8 es una matriz, y aplanado es una fila de 64 números. Mil imágenes, entonces, son una tabla de 64 columnas, una por píxel:

In [ ]:
digits = load_digits()
pixels = pd.DataFrame(digits.data, columns=[f"px_{i}" for i in range(64)])
pixels["label"] = digits.target
print("una imagen:", digits.images[0].shape, "→ una fila de", digits.data.shape[1], "columnas")
pixels.iloc[:3, [0, 1, 2, 3, 4, 5, 63, 64]]

**Un texto** se puede convertir en números de muchas formas. La más simple: elegir un **vocabulario** (una lista de palabras) y contar cuántas veces aparece cada una. Se llama **bolsa de palabras** (*bag of words*): se pierde el orden, pero cada texto queda como un vector del mismo largo. En la semana 20 veremos los *embeddings*, que conservan mucho más.

### Ejercicio 9: un texto como vector

Completa `text_to_counts(text, vocab)`: pasa `text` a minúsculas, sepáralo en palabras con `.split()` y devuelve un arreglo de NumPy de largo `len(vocab)` cuya posición `i` diga cuántas veces aparece `vocab[i]`. Las palabras que no están en el vocabulario se ignoran.

In [ ]:
vocab = ["el", "pingüino", "pesa", "nada", "mucho", "come", "pez"]


def text_to_counts(text, vocab):
    # TODO
    raise NotImplementedError

In [ ]:
def test_bag_of_words():
    v = text_to_counts("El pingüino nada mucho y come mucho pez", vocab)
    assert isinstance(v, np.ndarray), "devuelve un arreglo de NumPy"
    assert v.shape == (7,), f"forma {v.shape}, esperaba (7,)"
    assert v.tolist() == [1, 1, 0, 1, 2, 1, 1], f"obtuve {v.tolist()}"
    assert text_to_counts("hola mundo", vocab).sum() == 0, "las palabras fuera del vocabulario se ignoran"


check("Bolsa de palabras", test_bag_of_words)

Tabla, imagen o texto: el modelo siempre recibe un arreglo. Lo que cambia según el tipo de dato es **cómo** se convierte, y qué estructura conviene conservar. Esa pregunta guía las partes del curso dedicadas a visión (semana 13) y a texto (semana 20).

## 10. Cierre

**Hoy vimos:**

- `pd.read_csv`, `shape`, `info()`: lo primero con una tabla nueva. Columnas numéricas y categóricas.
- Filtrar con máscaras, como en NumPy: `df[df["col"] > valor]`, `&` y `|` con paréntesis. `value_counts()` para contar.
- Datos faltantes: `isna().sum()` para encontrarlos; `dropna` o `fillna` para tratarlos. NumPy no ignora un `NaN`.
- Media, mediana y desviación estándar. La mediana es robusta a los valores atípicos; pandas divide por $n-1$.
- `groupby`: separar, aplicar, juntar. Un número de toda la tabla puede esconder grupos muy distintos.
- Histogramas y dispersión, siempre con ejes, unidades y leyenda. La correlación mide una relación lineal, no una causa, y puede cambiar de signo por grupos (paradoja de Simpson).
- Todo modelo se compara con una línea base. Tabla, imagen o texto: al final, un arreglo.

**Para la semana 4:**

1. **Lectura previa** (MÓLÓ, en inglés): [Calculus basics for ML](https://colab.research.google.com/drive/1yc25B_ODIvC1TtdjAJllyaSfXXCOzgSB), [Functions and Parameters](https://colab.research.google.com/drive/1LvEAjw4XcV_k0QZ-QMkNTyaIS9y1ntPv) y [Probability and Distributions](https://colab.research.google.com/drive/1ssiR39Q8H0W7LTLSKR8VkayxqX7wpzWM).
2. **Cuota semanal** en [la plataforma](https://ioai.artix.tech/).
3. Si no terminaste los ejercicios de hoy, termínalos: el Sprint 0 (semanas 5 y 6) es limpiar y analizar un CSV desordenado, con estas mismas herramientas.

La semana 4 es una clase de cálculo y probabilidad: derivadas, gradiente y descenso de gradiente a mano.